In [13]:
import sys
from pathlib import Path

src_path = Path("/Users/tliu/oh_suppression/codes/src")

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from oh_suppression.coupling import *

In [14]:
from oh_suppression.point_source_coupling import *
from oh_suppression.flat_background_coupling import *
from oh_suppression.extended_source_coupling import *

In [15]:
import astropy.units as u
from astropy.modeling.models import Sersic2D
# Source: https://docs.astropy.org/en/stable/api/astropy.modeling.functional_models.Sersic2D.html

In [16]:
from matplotlib import pyplot as plt
import numpy as np
from scipy import special #import jv, kv
# from scipy.special import j1 # Bessel function of the first kind of order 1, for the diffraction pattern calculation
from scipy.optimize import root_scalar
from scipy.constants import c, epsilon_0, mu_0, pi
import scipy.integrate as integrate

In [17]:
# jn_zeros is a function that returns the zeros of the Bessel function of the first kind, 
# which are needed to find the roots for the LP01 mode in a step-index fibre
from scipy.special import jn_zeros  # NEW: needed to get the first zero of J0


In [18]:
from scipy.optimize import brentq
from scipy.optimize import fminbound

In [19]:
from dataclasses import dataclass
import pandas as pd

In [20]:
from joblib import Parallel, delayed
import os

In [21]:
def radial_to_eta_map(
        eta_psf_interp,
        decentre_array: np.ndarray,
        x: np.ndarray,
        y: np.ndarray
) -> np.ndarray:
    """
    Convert a radial point-source coupling efficiency function
    into a 2D coupling efficiency map.
    """

    # Calculate the radial distance from the fibre centre.
    r_grid = np.sqrt(x**2 + y**2)

    # Create an empty array for the 2D coupling map.
    eta_map_2d = np.zeros_like(r_grid)

    # Select positions covered by the calculated radial response.
    inside_range = r_grid <= decentre_array[-1]

    # Interpolate the radial coupling efficiency onto the 2D grid.
    eta_map_2d[inside_range] = eta_psf_interp(
        r_grid[inside_range]
    )

    return eta_map_2d

In [22]:
def generate_jitter_positions(
    sigma_jitter_rms_mas: float,
    focal_length: float,
    demagnification: float,
    n_samples: int,
    rng: np.random.Generator
) -> tuple[np.ndarray, np.ndarray]:
    """
    Generate random x and y tip-tilt jitter displacements at the fibre plane. 
    The displacements are drawn from a normal distribution with mean 0 and standard deviation corresponding to 
    the RMS jitter specified in milliarcseconds.
    This is a helper function to generate random jitter positions for Monte Carlo simulations of coupling efficiency.

    Parameters
    ----------
    sigma_jitter_rms_mas:
        RMS jitter in milliarcseconds.

    focal_length:
        Telescope focal length in metres.

    demagnification:
        Demagnification factor between the telescope focal plane
        and the fibre plane.

    n_samples:
        Number of random jitter positions.

    rng:
        NumPy random number generator.

    Returns
    -------
    delta_x, delta_y:
        Random x and y jitter displacements at the fibre plane, in metres.
    """

    # Convert RMS jitter from milliarcseconds to metres at the fibre plane.
    sigma_jitter_metres = sigma_jitter_milliarcsec_to_metres(
        sigma_jitter_rms_mas,
        focal_length,
        demagnification
    )

    # Generate random x jitter positions.
    delta_x = rng.normal(
        loc=0,
        scale=sigma_jitter_metres,
        size=n_samples
    )

    # Generate random y jitter positions.
    delta_y = rng.normal(
        loc=0,
        scale=sigma_jitter_metres,
        size=n_samples
    )

    return delta_x, delta_y

In [23]:
def Monte_Carlo_galaxy_source_coupling(
    x: np.ndarray,
    y: np.ndarray,
    x_1d: np.ndarray,
    y_1d: np.ndarray,
    delta_x: np.ndarray,
    delta_y: np.ndarray,
    eta_map_fixed: np.ndarray,
    galaxy_model,
    plate_scale_demag: float,
) -> np.ndarray:
    """
    Perform a Monte Carlo simulation to calculate the coupling efficiency
    of an extended galaxy source in the presence of tip-tilt jitter.

    For each random jitter position, the galaxy brightness distribution
    is shifted relative to the fixed fibre, and the resulting coupling
    efficiency is calculated using the fixed 2D coupling map.

    Parameters
    ----------
    x : np.ndarray
        2D array of x-coordinates at the fibre plane, in metres.

    y : np.ndarray
        2D array of y-coordinates at the fibre plane, in metres.

    x_1d : np.ndarray
        1D array of x-coordinates at the fibre plane, in metres.

    y_1d : np.ndarray
        1D array of y-coordinates at the fibre plane, in metres.

    delta_x : np.ndarray
        Random x-direction jitter displacements at the fibre plane,
        in metres.

    delta_y : np.ndarray
        Random y-direction jitter displacements at the fibre plane,
        in metres.

    eta_map_fixed : np.ndarray
        Fixed 2D point-source coupling efficiency map.

    galaxy_model : callable
        Galaxy surface-brightness model evaluated using angular
        coordinates.

    plate_scale_demag : float
        Plate scale at the demagnified fibre input plane, in arcsec/m.

    Returns
    -------
    eta_samples : np.ndarray
        Coupling efficiency for each Monte Carlo jitter position.
    """

    # Convert the fibre-plane coordinates from metres to arcseconds.
    x_arcsec = x * plate_scale_demag
    y_arcsec = y * plate_scale_demag

    # Number of Monte Carlo samples is determined by the number
    # of generated jitter positions.
    n_samples = len(delta_x)

    # Create an empty array in which to store the coupling
    # efficiency corresponding to each random displacement.
    eta_samples = np.empty(n_samples)

    # Loop over each Monte Carlo sample, applying the random jitter displacement to the galaxy model 
    # and calculating the resulting coupling efficiency.
    for i in range(n_samples):

        # Convert the random jitter displacement from metres at the
        # fibre plane to arcseconds on the galaxy angular grid.
        dx_arcsec = delta_x[i] * plate_scale_demag
        dy_arcsec = delta_y[i] * plate_scale_demag

        # Shift the galaxy brightness distribution relative to the
        # fixed fibre.
        img_jittered = galaxy_model(
            x_arcsec - dx_arcsec,
            y_arcsec - dy_arcsec
        )

        # Calculate the coupling efficiency of the displaced galaxy.
        eta_samples[i] = eta_extended_source_2d(
            eta_map_2d=eta_map_fixed,
            galaxy_brightness_2d=img_jittered,
            x_1d=x_1d,
            y_1d=y_1d
        )

    return eta_samples

## Define a function that sweeps the jitter affected coupling efficiency as a function of radius

In [ ]:
def jittered_ext_src_eta_vs_core_radius: 
    core_radius_array, 
    x, y, 